# SmartCare Hospital AI - Option C
## Explainable and Leakage-Aware Multiclass Disease-Risk Classification

**Target:** `disease_risk_level` (Low, Medium, High)  
**Prediction point:** Initial patient assessment, before admission, treatment, billing, or discharge.

## 1. Problem definition

SmartCare Hospital requires an interpretable machine-learning system that classifies patients into Low, Medium, and High disease-risk levels from structured information available during initial assessment. The system is intended as an educational decision-support prototype. It must not replace professional clinical judgement.

### Aim
Develop and evaluate a leakage-aware, explainable multiclass classification pipeline for the SmartCare dataset.

### Research questions
1. Which machine-learning algorithm achieves the strongest macro-F1 score?
2. Does diagnosis and hospital-history context improve a clinical-only model?
3. Which features influence Low, Medium, and High predictions?
4. Which class and class boundary cause the most errors?
5. How reliable are the predicted class probabilities?

### Research gap
The reviewed literature is dominated by disease-specific or population-specific studies, and several studies reuse the same small maternal-health dataset. Within that literature, limited work jointly examines a general-hospital three-level risk label with an explicit prediction-time feature audit, class-specific and ordinal error analysis, calibrated uncertainty, and both global and patient-level explanations. This project addresses that methodological gap on the supplied synthetic SmartCare dataset.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Display and plot settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

# Set file paths and check they exist
PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "smartcare_ai_dataset_1000.csv"
DICTIONARY_PATH = PROJECT_DIR / "smartcare_ai_dataset_data_dictionary.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")
if not DICTIONARY_PATH.exists():
    raise FileNotFoundError(f"Data dictionary not found: {DICTIONARY_PATH}")

print(f"Project directory: {PROJECT_DIR}")

Project directory: c:\Users\Anjana\OneDrive\Desktop\smartcare-ai-risk-classification


## 2. Load the dataset and data dictionary

In [2]:
# Load dataset and data dictionary
df = pd.read_csv(DATA_PATH)
data_dictionary = pd.read_csv(DICTIONARY_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Data dictionary entries: {len(data_dictionary)}")
display(df.head())
display(data_dictionary)

Dataset shape: 1,000 rows x 33 columns
Data dictionary entries: 33


,record_id,patient_id,age,gender,blood_group,department,diagnosis,appointment_date,waiting_days,previous_appointments,missed_previous_appointments,appointment_status,admitted,room_type,length_of_stay_days,previous_admissions,systolic_bp,diastolic_bp,blood_sugar_mg_dl,cholesterol_mg_dl,bmi,lab_tests_count,treatments_count,consultation_fee_lkr,room_charge_lkr,lab_charge_lkr,medicine_charge_lkr,total_bill_lkr,payment_status,payment_method,no_show,readmitted_30_days,disease_risk_level
0,1,P10001,53,Male,A-,General Medicine,Migraine,2025-04-10,10,1,0,Completed,0,NaN,0,1,127,75,117,211,26.1,0,3,2000,0,0,11596,13596,Paid,Insurance,0,0,High
1,2,P10002,26,Male,B-,General Medicine,Diabetes,2025-05-15,2,3,1,Completed,0,NaN,0,0,130,73,136,173,32.8,0,1,2000,0,0,3652,5652,Paid,Insurance,0,0,Medium
2,3,P10003,22,Male,B+,Orthopedics,Back Pain,2025-07-09,22,7,1,No-Show,0,NaN,0,1,141,64,90,176,29.4,1,0,2500,0,1200,2562,6262,Unpaid,Insurance,1,0,Medium
3,4,P10004,44,Female,AB-,Cardiology,Asthma,2025-10-16,16,1,0,Completed,0,NaN,0,0,124,82,126,189,24.9,2,1,2000,0,5000,10262,17262,Paid,Online,0,0,Medium
4,5,P10005,51,Female,O+,Neurology,Hypertension,2025-12-18,12,4,0,Scheduled,0,NaN,0,1,119,81,65,195,27.0,2,0,4000,0,6000,10414,20414,Paid,Cash,0,0,Medium


,Column,Description
0,record_id,Unique row identifier
1,patient_id,Synthetic patient identifier
2,age,Patient age in years
3,gender,Patient gender
4,blood_group,Patient blood group
5,department,Hospital department
6,diagnosis,Primary diagnosis category
7,appointment_date,Appointment date
8,waiting_days,Number of days between booking and appointment
9,previous_appointments,Number of previous appointments


## 3. Dataset structure and attribute types

In [3]:
# Summary of each column type, missing values, unique values
structure = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null": df.notna().sum().values,
    "missing": df.isna().sum().values,
    "missing_pct": (df.isna().mean().values * 100).round(2),
    "unique_values": df.nunique(dropna=True).values,
})
display(structure)

,column,dtype,non_null,missing,missing_pct,unique_values
0,record_id,int64,1000,0,0.0,1000
1,patient_id,object,1000,0,0.0,1000
2,age,int64,1000,0,0.0,88
3,gender,object,1000,0,0.0,2
4,blood_group,object,1000,0,0.0,8
5,department,object,1000,0,0.0,7
6,diagnosis,object,1000,0,0.0,10
7,appointment_date,object,1000,0,0.0,339
8,waiting_days,int64,1000,0,0.0,45
9,previous_appointments,int64,1000,0,0.0,10


In [4]:
# Split columns into numeric and categorical
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(exclude=np.number).columns.tolist()

print(f"Numeric columns ({len(numeric_columns)}): {numeric_columns}")
print(f"Categorical/date columns ({len(categorical_columns)}): {categorical_columns}")
display(df[numeric_columns].describe().T.round(2))

Numeric columns (22): ['record_id', 'age', 'waiting_days', 'previous_appointments', 'missed_previous_appointments', 'admitted', 'length_of_stay_days', 'previous_admissions', 'systolic_bp', 'diastolic_bp', 'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi', 'lab_tests_count', 'treatments_count', 'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr', 'total_bill_lkr', 'no_show', 'readmitted_30_days']
Categorical/date columns (11): ['patient_id', 'gender', 'blood_group', 'department', 'diagnosis', 'appointment_date', 'appointment_status', 'room_type', 'payment_status', 'payment_method', 'disease_risk_level']


,count,mean,std,min,25%,50%,75%,max
record_id,1000.0,500.50,288.82,1.0,250.75,500.5,750.25,1000.0
age,1000.0,44.74,17.85,1.0,33.00,44.0,57.00,90.0
waiting_days,1000.0,21.85,13.04,0.0,11.00,22.0,34.00,44.0
previous_appointments,1000.0,2.88,1.69,0.0,2.00,3.0,4.00,10.0
missed_previous_appointments,1000.0,0.55,0.74,0.0,0.00,0.0,1.00,4.0
admitted,1000.0,0.33,0.47,0.0,0.00,0.0,1.00,1.0
length_of_stay_days,1000.0,1.10,1.89,0.0,0.00,0.0,2.00,9.0
previous_admissions,1000.0,0.86,0.96,0.0,0.00,1.0,1.00,5.0
systolic_bp,1000.0,128.42,15.49,85.0,117.00,128.0,139.00,178.0
diastolic_bp,1000.0,78.83,10.04,50.0,72.00,79.0,86.00,111.0
